# llmpic V0.3.0 — 自然语言 → 专业图表

> 说人话，出图表。**12 种类型。** PNG / SVG / PDF 导出。Jupyter 内联显示。异步批量。迭代编辑。地理地图。安全沙箱。

[GitHub](https://github.com/ADW-19/llmpic) · [文档](https://ADW-19.github.io/llmpic/) · [PyPI](https://pypi.org/project/llmpic/)

## 1. 初始化 SDK

兼容任意 OpenAI 标准接口：OpenAI / DeepSeek / Azure / 智谱 GLM / Ollama / vLLM...

In [ ]:
import os
from llmpic import llmPIC

lp = llmPIC(
    api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
    base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
    model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    max_tokens=4096,
)
print(f"SDK 就绪 — 模型: {lp.model}")

## 2. 快速入门 — 一句话出图

不传任何数据，LLM 会用 numpy 自动生成逼真的演示数据。

In [ ]:
r = lp.plot("过去12个月的月度销售额变化趋势").render()

if r.success:
    r.show()  # Jupyter cell 下方直接出图
    print(f"大小: {r.size_kb:.0f}KB | Token: 输入={r.token_usage['input']} 输出={r.token_usage['output']}")
else:
    print(f"失败: {r.error_message}")

## 3. 数据输入 — 6 种方式

llmpic 几乎接受所有数据格式。LLM 会收到序列化后的数据摘要（列名、类型、前几行），然后使用你真实的列名写代码。

### 3.1 不传数据 — 自动生成

In [ ]:
lp.plot("30天内CPU使用率变化，每天两个峰值").render().show()

### 3.2 内联数据 — 数字直接写在 query 里

In [ ]:
lp.bar("Q1预算: 研发=200K, 市场=150K, 销售=180K, 人事=100K").render().show()

### 3.3 pandas DataFrame — 自动识别列名

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "月份": pd.date_range("2025-01-01", periods=12, freq="MS").strftime("%m月"),
    "营收": np.random.randint(80, 200, 12),
    "成本": np.random.randint(50, 120, 12),
})
df["利润"] = df["营收"] - df["成本"]

lp.bar("各月营收、成本、利润对比，分组柱状图").data(df).render().show()

### 3.4 NumPy 数组

In [ ]:
data = np.random.randn(1000)
lp.hist("考试成绩分布，30个区间，叠加 KDE 密度曲线").data(data).render().show()

### 3.5 Python 字典 / 列表

In [ ]:
lp.pie("市场份额分布").data({
    "产品A": 35, "产品B": 28, "产品C": 18, "产品D": 12, "其他": 7
}).render().show()

lp.plot("一周气温变化").data([22, 24, 19, 26, 28, 25, 23]).render().show()

### 3.6 CSV / Excel — 通过 pandas 读取

In [ ]:
# df = pd.read_csv("你的数据.csv")
# lp.scatter("年龄与收入的相关性分析").data(df).render().show()

## 4. 自定义风格

6 种预设配色 + 10 个样式参数。在 `.render()` 前链式调用 `.style()`。

### 4.1 预设配色方案

In [ ]:
from IPython.display import display, Markdown

for scheme in ["blues", "warm", "cool", "pastel", "dark", "grayscale"]:
    r = lp.bar(f"Q4产品营收: 云服务=450, AI=320, 安全=280").style({"color_scheme": scheme}).render()
    if r.success:
        display(Markdown(f"**{scheme}**"))
        r.show()

### 4.2 全部样式参数

In [ ]:
lp.plot("12个月营收趋势").data(df).style({
    "figsize": [14, 7],         # 图表尺寸（英寸）宽×高
    "dpi": 200,                  # 输出分辨率
    "color_scheme": "blues",     # 配色: blues | warm | cool | pastel | dark | grayscale
    "title_fontsize": 18,        # 标题字号
    "label_fontsize": 14,        # 坐标轴标签字号
    "tick_fontsize": 12,         # 刻度字号
    "grid": True,                # 显示背景网格
    "grid_alpha": 0.3,           # 网格线透明度（0-1）
    "tight_layout": True,        # 自动调整布局避免裁剪
    "facecolor": "#FAFAFA",      # 图表背景色
}).render().show()

## 5. 全部 12 种图表类型

In [ ]:
from IPython.display import display, Markdown

charts = [
    ("1. plot — 折线图", lp.plot("2024年每月营收趋势，12个月")),
    ("2. scatter — 散点图", lp.scatter("随机散点图：50个点，年龄vs收入，带趋势线")),
    ("3. bar — 柱状图", lp.bar("部门预算: 研发=200K, 市场=150K, 销售=180K, 人事=100K")),
    ("4. pie — 饼图", lp.pie("市场份额: A=40%, B=25%, C=20%, 其他=15%")),
    ("5. hist — 直方图", lp.hist("正态分布 N(0,1)，1000个样本，叠加KDE密度曲线")),
    ("6. heatmap — 热力图", lp.heatmap("6x6 相关性矩阵，标注数值，coolwarm配色")),
    ("7. boxplot — 箱线图", lp.boxplot("四组成绩对比: 对照组, 实验A, 实验B, 实验C（每组30个）")),
    ("8. area — 面积图", lp.area("2020-2024三条产品线收入构成，堆叠面积图")),
    ("9. radar — 雷达图", lp.radar("产品评分: 性能=4, 易用=3, 稳定=5, 价格=2, 售后=4")),
    ("10. subplots — 仪表盘", lp.subplots("2x2看板: 销售折线, 地区柱状, 客户散点, 增长直方图")),
    ("11. custom — 智能推荐", lp.custom("分析用户留存率变化趋势及影响因素")),
    ("12. map — 地理地图 (v0.3.0)", lp.map("世界主要城市人口分布，Blues配色")),
]

for title, builder in charts:
    r = builder.render()
    display(Markdown(f"### {title}"))
    if r.success:
        r.show()
        print(f"   {r.size_kb:.0f}KB | Token: 入={r.token_usage['input']} 出={r.token_usage['output']}")
    else:
        print(f"   失败: {r.error_message[:120]}")

## 6. ChartResult — 查看详情 & 导出

### 6.1 基本属性

In [ ]:
r = lp.plot("sin(x) 0到2π，平滑曲线").render()

print(f"是否成功:     {r.success}")
print(f"图片大小:     {r.size_kb:.1f} KB")
print(f"输出格式:     {r._format}")
print(f"输入 Token:   {r.token_usage['input']}")
print(f"输出 Token:   {r.token_usage['output']}")
print(f"代码长度:     {len(r.code)} 字符")

print(f"\n--- 生成的代码（前500字符） ---\n{r.code[:500]}...")

### 6.2 保存到文件 — PNG / SVG / PDF

In [ ]:
r = lp.plot("24小时CPU使用率，上午9点和下午3点峰值").render()

path1 = r.save("cpu_usage.png")
path2 = r.save("cpu_usage.svg")
path3 = r.save("cpu_usage.pdf")
path4 = r.save()  # 默认 ~/llmpic_charts/chart_{时间戳}.png

print(f"已保存:\n  {path1}\n  {path2}\n  {path3}\n  {path4}")

### 6.3 Base64 编码 — 网页嵌入

In [ ]:
r = lp.pie("市场份额: A=40%, B=30%, C=20%, D=10%").render()

png_b64 = r.base64()
svg_b64 = r.base64_svg()

print(f"PNG base64: {len(png_b64):,} 字符")
print(f"SVG base64: {len(svg_b64):,} 字符")

### 6.4 懒加载格式转换 — 同一结果转 SVG/PDF

In [ ]:
r = lp.plot("月度销售趋势").render()  # 默认 PNG

svg_bytes = r.svg_bytes   # 首次访问 → 重新渲染为 SVG → 缓存
pdf_bytes = r.pdf_bytes   # 首次访问 → 重新渲染为 PDF → 缓存
svg_str   = r.svg         # SVG 字符串

print(f"PNG:  {r.size_kb:.0f} KB")
print(f"SVG:  {len(svg_bytes) / 1024:.0f} KB")
print(f"PDF:  {len(pdf_bytes) / 1024:.0f} KB")

r.show()

## 7. 迭代编辑 — 自然语言逐步打磨

`.edit()` 将当前代码 + 修改请求发送给 LLM。返回**新的** ChartResult，原对象不变。

In [ ]:
from IPython.display import display, Markdown

v1 = lp.plot("季度销售: Q1=100, Q2=150, Q3=120, Q4=180").render()
display(Markdown("### v1 — 初始折线图"))
v1.show()

v2 = v1.edit("改成柱状图，用蓝色系")
display(Markdown("### v2 — 柱状图 + 蓝色"))
v2.show()

v3 = v2.edit("标题改为'2025年度销售报告'，标题字号加大到18，添加网格线")
display(Markdown("### v3 — 标题+网格"))
v3.show()

v4 = v3.edit("换成暖色系，Y轴加标签'营收（万元）'")
display(Markdown("### v4 — 最终版"))
v4.show()

v4.save("最终报告.png")
print("已保存: 最终报告.png")

## 8. 安全配置

两种安全模式。沙箱已阻断所有实际执行路径——**fast** 模式生产环境足够安全。

In [ ]:
# 快速模式（默认）— 32条预编译正则，几乎零延迟
lp_fast = llmPIC(
    api_key="sk-...", base_url="https://api...",
    safety_level="fast",
)

# 完整模式 — 正则 + LLM 语义审查，每张图额外 1-2s
lp_full = llmPIC(
    api_key="sk-...", base_url="https://api...",
    safety_level="full",
)

print("两种模式就绪。fast=仅正则, full=正则+LLM审查。")

## 9. 异步批量 — 并发生成

所有图表并行生成。总耗时 ≈ 最慢的那张。

In [ ]:
from llmpic import AsyncllmPIC
from IPython.display import display, Markdown
import time

async def run_batch():
    lp_async = AsyncllmPIC(
        api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
        base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
        model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    )

    t0 = time.time()
    results = await lp_async.batch([
        ("plot",     "全国12个月销售趋势"),
        ("bar",      "各地区销售额对比"),
        ("pie",      "市场份额分布"),
        ("scatter",  "客户年龄 vs 年消费额"),
        ("heatmap",  "5x5 特征相关性矩阵"),
    ])

    for i, r in enumerate(results):
        if r.success:
            display(Markdown(f"### batch[{i}] — {r.size_kb:.0f}KB"))
            r.show()
        else:
            print(f"[{i}] 失败: {r.error_message[:100]}")

    print(f"\n5张图并发生成，总耗时: {time.time()-t0:.1f}s")

await run_batch()

## 10. 进阶 — 智能推荐 & 查看生成代码

让 LLM 自动判断最佳图表类型。获取生成的 matplotlib 代码进行手动调优。

In [ ]:
r = lp.custom("分析用户留存率变化趋势及影响因素").render()
r.show()

print("=== 生成的代码 ===")
print(r.code)

print(f"\n=== Token 用量 ===")
print(f"输入: {r.token_usage['input']}, 输出: {r.token_usage['output']}")

## 11. 格式转换 — 一次调用，三种格式

一次 LLM 调用，三种格式输出。从已存储代码懒加载渲染。

In [ ]:
r = lp.bar("各区域销售额: 华北=320, 华南=280, 华东=260, 华西=200").render()

print(f"PNG: {r.size_kb:.0f}KB ({len(r.image_bytes):,} 字节)")
print(f"SVG: {len(r.svg_bytes) / 1024:.0f}KB")
print(f"PDF: {len(r.pdf_bytes) / 1024:.0f}KB")
print(f"\n生成了 {len(r.code)} 字符的 matplotlib 代码，渲染为 3 种格式")

---

**以上就是全部！** 你已经覆盖了 llmpic V0.3.0 的所有 API：

| API | 章节 |
|-----|------|
| `llmPIC()`, `AsyncllmPIC()` | 1, 9 |
| `.plot()` `.scatter()` `.bar()` `.pie()` `.hist()` `.heatmap()` `.boxplot()` `.area()` `.radar()` `.map()` `.subplots()` `.custom()` | 2, 5 |
| `.data()` | 3 |
| `.style()` | 4 |
| `.render()` `.save()` | 2, 6 |
| `.show()` `.base64()` `.base64_svg()` | 2, 6 |
| `.edit()` | 7 |
| `safety_level` | 8 |
| `.batch()` | 9 |
| `.code` `.token_usage` `.size_kb` `.svg_bytes` `.pdf_bytes` `.svg` | 6, 10 |

⭐ [GitHub](https://github.com/ADW-19/llmpic) · [文档](https://ADW-19.github.io/llmpic/) · [PyPI](https://pypi.org/project/llmpic/)